In [ ]:
#!/usr/bin/env python3
"""
Generate Figure 1 for the Falcon 9 / BCHH paper.

Outputs:
  figures/fig01_slc40_bchh_location.png
  figures/fig01_slc40_bchh_location.pdf
"""

from pathlib import Path

import contextily as cx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FormatStrFormatter, FuncFormatter, MaxNLocator
from pyproj import Geod, Transformer


OUTDIR = Path("/Users/thompsong/Library/CloudStorage/Box-Box/thompsong/3_Project_Documents/NASAprojects/201602_Rocket_Seismology/writing_papers_reports")
OUTDIR.mkdir(exist_ok=True)


## Launchpad (and Camera) Coordinates

In [ ]:
KML_FILE = Path("launchpads_cameras.kml")
from read_launchpads_from_kml import read_kml_points
kml_points = read_kml_points(KML_FILE)
kml_df = pd.DataFrame.from_dict(kml_points, orient="index").reset_index(drop=True)

# Can either extract dicts like this:
SLC40 = kml_points["SLC40"]
SLC41 = kml_points["SLC41"]

# Or like:
#SLC40 = kml_df.loc[kml_df["name"] == "SLC40"].iloc[0]

In [ ]:
BCHH_SENSORS = pd.DataFrame(
    [
        ["Seismometer", "BCHH\nSeismometer\n(HHZ/N/E)", 541820.43, 3160866.50, 28.5740171, -80.572375],
        ["HD1", "HD1\n(Infrasound 1)", 541816.30, 3160889.14, 28.574222, -80.572417],
        ["HD2", "HD2\n(Infrasound 2)", 541828.28, 3160851.47, 28.573881, -80.572296],
        ["HD3", "HD3\n(Infrasound 3)", 541802.34, 3160865.25, 28.574006, -80.572560],
    ],
    columns=["sensor", "label", "easting", "northing", "lat", "lon"],
)
print(BCHH_SENSORS)

In [ ]:

BCHH = {
    "name": "BCHH",
    "lat": float(BCHH_SENSORS.loc[BCHH_SENSORS.sensor == "Seismometer", "lat"].iloc[0]),
    "lon": float(BCHH_SENSORS.loc[BCHH_SENSORS.sensor == "Seismometer", "lon"].iloc[0]),
}

GEOD = Geod(ellps="WGS84")
LL_TO_UTM = Transformer.from_crs("EPSG:4326", "EPSG:32617", always_xy=True)
UTM_TO_LL = Transformer.from_crs("EPSG:32617", "EPSG:4326", always_xy=True)




In [ ]:
from pathlib import Path
from obspy import UTCDateTime

from xml_extract import (
    inventory_stations_to_dataframe,
    load_station_sensor_dataframe,
)

STATIONXML_FILE = Path("../03_stationxml/KSC.xml")

STARTTIME = UTCDateTime("2016-09-01T00:00:00")
ENDTIME = UTCDateTime("2016-09-02T00:00:00")

NETWORK_CODE = "1R"
STATION_CODE = "BCHH"
LOCATION_CODE = "10"

SOURCE_CRS = "EPSG:4326"
TARGET_CRS = "EPSG:32617"  # WGS84 / UTM zone 17N


CHANNEL_TO_SENSOR = {
    "DHZ": "Seismometer",
    "DHN": "Seismometer",
    "DHE": "Seismometer",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
}

CHANNEL_TO_LABEL = {
    "DHZ": "BCHH",
    "DHN": "BCHH",
    "DHE": "BCHH",
    "DD1": "HD1",
    "DD2": "HD2",
    "DD3": "HD3",
}

inventory_event, channels_df, BCHH_SENSORS = (
    load_station_sensor_dataframe(
        stationxml_file=STATIONXML_FILE,
        starttime=STARTTIME,
        endtime=ENDTIME,
        network_code=NETWORK_CODE,
        station_code=STATION_CODE,
        location_code=LOCATION_CODE,
        channel_to_sensor=CHANNEL_TO_SENSOR,
        channel_to_label=CHANNEL_TO_LABEL,
        source_crs=SOURCE_CRS,
        target_crs=TARGET_CRS,
    )
)

stations_df = inventory_stations_to_dataframe(
    inventory=inventory_event
)

print(inventory_event)

display(stations_df)
display(channels_df)
display(BCHH_SENSORS)

In [ ]:

# -----------------------------------------------------------------------------
# Helper functions
# -----------------------------------------------------------------------------


def equal_scale_lonlat_limits(center_lon, center_lat, width_lon=None, height_lat=None):
    """Return lon/lat limits with approximately equal ground scale on x and y."""
    coslat = np.cos(np.deg2rad(center_lat))
    if width_lon is None and height_lat is None:
        raise ValueError("Provide either width_lon or height_lat")
    if width_lon is not None:
        height_lat = width_lon * coslat
    else:
        width_lon = height_lat / coslat
    return (
        (center_lon - width_lon / 2, center_lon + width_lon / 2),
        (center_lat - height_lat / 2, center_lat + height_lat / 2),
    )


def set_equal_ground_aspect(ax):
    """Make one plotted cm represent roughly the same ground distance E-W and N-S."""
    lat_mid = np.mean(ax.get_ylim())
    ax.set_aspect(1 / np.cos(np.deg2rad(lat_mid)), adjustable="box")


def add_basemap(ax, provider="voyager", alpha=0.85, zoom="auto"):
    if provider == "osm":
        source = cx.providers.OpenStreetMap.Mapnik
    elif provider == "positron":
        source = cx.providers.CartoDB.Positron
    else:
        source = cx.providers.CartoDB.Voyager

    cx.add_basemap(
        ax,
        source=source,
        crs="EPSG:4326",
        reset_extent=False,
        attribution_size=7,
        alpha=alpha,
        zoom=zoom,
    )


def plain_utm_formatter(x, pos):
    return f"{x:.0f}"


def setup_dual_axes(
    ax,
    lonlim,
    latlim,
    lonfmt="%.3f",
    latfmt="%.3f",
    xlabel="Longitude (°W)",
    ylabel="Latitude (°N)",
    top_xlabel="Easting (m, UTM 17N)",
    right_ylabel="Northing (m, UTM 17N)",
    top_nbins=5,
    right_nbins=6,
):
    ax.set_xlim(lonlim)
    ax.set_ylim(latlim)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.xaxis.set_major_formatter(FormatStrFormatter(lonfmt))
    ax.yaxis.set_major_formatter(FormatStrFormatter(latfmt))
    ax.xaxis.get_offset_text().set_visible(False)
    ax.yaxis.get_offset_text().set_visible(False)

    lon_mid = float(np.mean(lonlim))
    lat_mid = float(np.mean(latlim))
    e_mid, n_mid = LL_TO_UTM.transform(lon_mid, lat_mid)

    def lon_to_easting(lon):
        lon = np.asarray(lon, dtype=float)
        e, _ = LL_TO_UTM.transform(lon, np.full_like(lon, lat_mid))
        return e

    def easting_to_lon(e):
        e = np.asarray(e, dtype=float)
        lon, _ = UTM_TO_LL.transform(e, np.full_like(e, n_mid))
        return lon

    def lat_to_northing(lat):
        lat = np.asarray(lat, dtype=float)
        _, n = LL_TO_UTM.transform(np.full_like(lat, lon_mid), lat)
        return n

    def northing_to_lat(n):
        n = np.asarray(n, dtype=float)
        _, lat = UTM_TO_LL.transform(np.full_like(n, e_mid), n)
        return lat

    ax_top = ax.secondary_xaxis("top", functions=(lon_to_easting, easting_to_lon))
    ax_top.set_xlabel(top_xlabel)
    ax_top.xaxis.set_major_formatter(FuncFormatter(plain_utm_formatter))
    ax_top.xaxis.set_major_locator(MaxNLocator(nbins=top_nbins, integer=True))
    ax_top.xaxis.get_offset_text().set_visible(False)

    ax_right = ax.secondary_yaxis("right", functions=(lat_to_northing, northing_to_lat))
    ax_right.set_ylabel(right_ylabel)
    ax_right.yaxis.set_major_formatter(FuncFormatter(plain_utm_formatter))
    ax_right.yaxis.set_major_locator(MaxNLocator(nbins=right_nbins, integer=True))
    ax_right.yaxis.get_offset_text().set_visible(False)

    return ax_top, ax_right


def label(ax, lon, lat, text, dx=0.0002, dy=0.00015, size=9, ha="left", va="center"):
    ax.text(
        lon + dx,
        lat + dy,
        text,
        fontsize=size,
        ha=ha,
        va=va,
        bbox=dict(facecolor="white", edgecolor="0.8", alpha=0.90, pad=1.4),
        zorder=30,
    )


def add_north_arrow_axes(ax, xy=(0.93, 0.78), length=0.10):
    x, y = xy
    ax.annotate(
        "",
        xy=(x, y + length),
        xytext=(x, y),
        xycoords="axes fraction",
        arrowprops=dict(facecolor="black", edgecolor="black", width=4, headwidth=13),
        zorder=50,
        clip_on=True,
    )
    ax.text(
        x,
        y + length + 0.018,
        "N",
        transform=ax.transAxes,
        ha="center",
        va="bottom",
        fontsize=10,
        fontweight="bold",
        zorder=51,
        clip_on=True,
    )


def add_scalebar_lonlat(ax, lon, lat, length_m, label_text):
    lon2, lat2, _ = GEOD.fwd(lon, lat, 90, length_m)
    tick_h = 0.018 * (ax.get_ylim()[1] - ax.get_ylim()[0])

    ax.plot([lon, lon2], [lat, lat2], "k-", lw=3, solid_capstyle="butt", zorder=40)
    ax.plot([lon, lon], [lat - tick_h / 2, lat + tick_h / 2], "k-", lw=2, zorder=40)
    ax.plot([lon2, lon2], [lat - tick_h / 2, lat + tick_h / 2], "k-", lw=2, zorder=40)
    ax.text(
        (lon + lon2) / 2,
        lat + 1.15 * tick_h,
        label_text,
        ha="center",
        va="bottom",
        fontsize=9,
        bbox=dict(facecolor="white", edgecolor="none", alpha=0.85, pad=1),
        zorder=41,
    )


# -----------------------------------------------------------------------------
# Figure construction
# -----------------------------------------------------------------------------


def make_figure():
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(12.4, 5.8),
        gridspec_kw={"width_ratios": [1.0, 1.12]},
    )
    fig.subplots_adjust(left=0.065, right=0.975, bottom=0.105, top=0.89, wspace=0.48)

    # ------------------------------------------------------------------
    # Panel A: SLC-40 to BCHH geometry
    # ------------------------------------------------------------------
    ax = axes[0]
    ax.set_title("A", loc="left", fontsize=14, fontweight="bold")

    center_lon = 0.5 * (SLC40["lon"] + BCHH["lon"]) - 0.003
    center_lat = 0.5 * (SLC40["lat"] + BCHH["lat"]) + 0.0015
    lonlim, latlim = equal_scale_lonlat_limits(center_lon, center_lat, width_lon=0.037)

    setup_dual_axes(
        ax,
        lonlim,
        latlim,
        lonfmt="%.3f",
        latfmt="%.3f",
        right_ylabel="",
        top_nbins=5,
        right_nbins=5,
    )
    set_equal_ground_aspect(ax)
    add_basemap(ax, provider="voyager", alpha=0.82, zoom="auto")
    ax.set_xlim(lonlim)
    ax.set_ylim(latlim)

    az, _, dist = GEOD.inv(SLC40["lon"], SLC40["lat"], BCHH["lon"], BCHH["lat"])
    ax.plot([SLC40["lon"], BCHH["lon"]], [SLC40["lat"], BCHH["lat"]], "k-", lw=2.2, zorder=10)

    ax.scatter(SLC40["lon"], SLC40["lat"], marker="^", s=185, c="yellow", edgecolor="black", zorder=20)
    ax.scatter(SLC41["lon"], SLC41["lat"], marker="^", s=145, c="white", edgecolor="black", zorder=20)
    ax.scatter(BCHH["lon"], BCHH["lat"], marker="o", s=145, c="black", edgecolor="black", zorder=20)

    label(ax, SLC40["lon"], SLC40["lat"], "SLC-40", dx=0.00028, dy=0.00025, size=9)
    label(ax, SLC41["lon"], SLC41["lat"], "SLC-41", dx=0.00028, dy=0.00025, size=9)
    label(ax, BCHH["lon"], BCHH["lat"], "BCHH", dx=0.00035, dy=0.00015, size=9)

    midlon = 0.5 * (SLC40["lon"] + BCHH["lon"])
    midlat = 0.5 * (SLC40["lat"] + BCHH["lat"])
    label(ax, midlon, midlat, f"{dist/1000:.2f} km\naz. {az:.0f}°", dx=0.0010, dy=0.00015, size=9)

    ax.text(-80.5920, 28.5662, "Indian\nRiver\nLagoon", fontsize=8, color="tab:blue", style="italic", zorder=35)
    ax.text(-80.5660, 28.5640, "Atlantic\nOcean", fontsize=8, color="tab:blue", style="italic", zorder=35)

    add_scalebar_lonlat(ax, lonlim[0] + 0.0065, latlim[0] + 0.0043, 1000, "1 km")
    add_north_arrow_axes(ax, xy=(0.93, 0.80), length=0.095)

    # ------------------------------------------------------------------
    # Panel B: BCHH array geometry
    # ------------------------------------------------------------------
    ax = axes[1]
    ax.set_title("B", loc="left", fontsize=14, fontweight="bold")

    center_lon = BCHH["lon"] + 0.00002
    center_lat = BCHH["lat"] + 0.00010
    lonlim, latlim = equal_scale_lonlat_limits(center_lon, center_lat, width_lon=0.00090)

    setup_dual_axes(
        ax,
        lonlim,
        latlim,
        lonfmt="%.5f",
        latfmt="%.5f",
        ylabel="",
        top_nbins=5,
        right_nbins=6,
    )
    set_equal_ground_aspect(ax)
    add_basemap(ax, provider="osm", alpha=0.72, zoom="auto")
    ax.set_xlim(lonlim)
    ax.set_ylim(latlim)

    infra = BCHH_SENSORS[BCHH_SENSORS.sensor.str.startswith("HD")]
    tri = infra.set_index("sensor").loc[["HD1", "HD2", "HD3", "HD1"]]
    ax.plot(tri["lon"], tri["lat"], color="0.45", lw=1.0, zorder=10)

    for _, row in BCHH_SENSORS.iterrows():
        if row.sensor == "Seismometer":
            ax.scatter(row.lon, row.lat, marker="s", s=145, c="black", edgecolor="black", zorder=20)
        else:
            ax.scatter(row.lon, row.lat, marker="o", s=115, c="white", edgecolor="black", zorder=20)

    for _, row in BCHH_SENSORS.iterrows():
        if row.sensor == "HD1":
            label(ax, row.lon, row.lat, row.label, dx=0.00001, dy=0.00003, size=8)
        elif row.sensor == "HD2":
            label(ax, row.lon, row.lat, row.label, dx=0.000015, dy=-0.00002, size=8)
        elif row.sensor == "HD3":
            label(ax, row.lon, row.lat, row.label, dx=-0.00004, dy=-0.00003, size=8)
        else:
            label(ax, row.lon, row.lat, row.label, dx=0.000015, dy=0.000035, size=8)

    add_scalebar_lonlat(ax, lonlim[0] + 0.00015, latlim[0] + 0.00007, 30, "30 m")
    add_north_arrow_axes(ax, xy=(0.23, 0.80), length=0.10)

    legend_handles = [
        plt.Line2D([], [], marker="s", ls="", ms=8, mfc="black", mec="black", label="Seismometer (3C)"),
        plt.Line2D([], [], marker="o", ls="", ms=8, mfc="white", mec="black", label="Infrasound (HD)"),
    ]
    ax.legend(handles=legend_handles, loc="lower right", fontsize=8, framealpha=0.9)

    return fig, axes


fig, axes = make_figure()
for ext in ["png", "pdf"]:
    fig.savefig(OUTDIR / f"fig01_slc40_bchh_location.{ext}", dpi=300, bbox_inches="tight", pad_inches=0.05)
plt.show()
